In [ ]:
!pip install pynput

In [ ]:
import numpy as np
import pandas as pd
import math
import time
from collections import deque
from pynput import keyboard
from pynput.keyboard import Key
import pygame
import heapq

pygame.init() 

In [ ]:

ppmm = 142 / 25.4 # 142 ppi of current display

height = int(ppmm*50) # 50mm box requirement
width = int(ppmm*180)

surface = pygame.display.set_mode((width, height))  # 50mm x 180mm
pygame.display.set_caption("A* Search Visualization")

obstacle_color = (255, 192, 203)
char_color = (255, 80, 200)

# visualization colors
visited_color = (0, 0, 255) #blue 
path_color = (255, 255, 0) # 
start_color = (0, 255, 0) # green
goal_color = (255, 0, 0) # red

# fill free space with black background
surface.fill((0, 0, 0))

# border
# these are the mathematical models of the letter geometry for the obstacle space
# J
J_rect = pygame.draw.rect(surface, obstacle_color, pygame.Rect(117, 39, 54, 122), width=30) # basic geometry of rectangle that forms "J"
J_arc = pygame.draw.arc(surface, obstacle_color, pygame.Rect(49, 89, 122, 122), start_angle=2.90, stop_angle=0, width=60) 
# S
S_arc = pygame.draw.arc(surface, obstacle_color, pygame.Rect(169, 39, 112, 112), start_angle=0.1, stop_angle=4.81239, width=60)
S_arc_2 = pygame.draw.arc(surface, obstacle_color, pygame.Rect(169, 94, 122, 122), start_angle=-3.24, stop_angle=1.97, width=60)
# 4432
four_rect = pygame.draw.rect(surface, obstacle_color, pygame.Rect(274, 39, 54, 97), width=30)
four_rect2 = pygame.draw.rect(surface, obstacle_color, pygame.Rect(274, 89, 97, 54), width=30)
four_rect3 = pygame.draw.rect(surface, obstacle_color, pygame.Rect(324, 39, 54, 172), width=30)
# 4
four_rect4 = pygame.draw.rect(surface, obstacle_color, pygame.Rect(374, 39, 54, 97), width=30)
four_rect5 = pygame.draw.rect(surface, obstacle_color, pygame.Rect(374, 89, 97, 54), width=30)
four_rect6 = pygame.draw.rect(surface, obstacle_color, pygame.Rect(424, 39, 54, 172), width=30)
# 3
three_rect = pygame.draw.arc(surface, obstacle_color, pygame.Rect(464, 39, 112, 112), start_angle=4.71239, stop_angle=2.85619, width=60)
three_rect2 = pygame.draw.rect(surface, obstacle_color, pygame.Rect(494, 97, 62, 54), width=30)
three_rect3 = pygame.draw.arc(surface, obstacle_color, pygame.Rect(464, 99, 112, 112), start_angle=3.42699, stop_angle=1.5708, width=60)
# 2
two_rect = pygame.draw.rect(surface, obstacle_color, pygame.Rect(589, 39, 102, 54), width=30)
two_rect2 = pygame.draw.rect(surface, obstacle_color, pygame.Rect(649, 39, 54, 114), width=30)
two_rect3 = pygame.draw.rect(surface, obstacle_color, pygame.Rect(624, 99, 54, 54), width=30)
two_rect4 = pygame.draw.rect(surface, obstacle_color, pygame.Rect(599, 129, 54, 54), width=30)
two_rect5 = pygame.draw.rect(surface, obstacle_color, pygame.Rect(579, 159, 142, 54), width=30)

# characters
# J
J_rect = pygame.draw.rect(surface, char_color, pygame.Rect(128, 50, 32, 100), width=30)
J_arc = pygame.draw.arc(surface, char_color, pygame.Rect(60, 100, 100, 100), start_angle=3.1415, stop_angle=0, width=30)
# S
S_arc = pygame.draw.arc(surface, char_color, pygame.Rect(180, 50, 90, 90), start_angle=0.5, stop_angle=4.71239, width=30)
S_arc_2 = pygame.draw.arc(surface, char_color, pygame.Rect(180, 110, 90, 90), start_angle=-2.64, stop_angle=1.57, width=30)
# 4432
four_rect = pygame.draw.rect(surface, char_color, pygame.Rect(285, 50, 32, 75), width=30)
four_rect2 = pygame.draw.rect(surface, char_color, pygame.Rect(285, 100, 75, 32), width=30)
four_rect3 = pygame.draw.rect(surface, char_color, pygame.Rect(335, 50, 32, 150), width=30)
# 4
four_rect4 = pygame.draw.rect(surface, char_color, pygame.Rect(385, 50, 32, 75), width=30)
four_rect5 = pygame.draw.rect(surface, char_color, pygame.Rect(385, 100, 75, 32), width=30)
four_rect6 = pygame.draw.rect(surface, char_color, pygame.Rect(435, 50, 32, 150), width=30)
# 3
three_rect = pygame.draw.arc(surface, char_color, pygame.Rect(475, 50, 90, 90), start_angle=4.71239, stop_angle=2.35619, width=30)
three_rect2 = pygame.draw.rect(surface, char_color, pygame.Rect(505, 108, 40, 32), width=30)
three_rect3 = pygame.draw.arc(surface, char_color, pygame.Rect(475, 110, 90, 90), start_angle=3.92699, stop_angle=1.5708, width=30)
# 2
two_rect = pygame.draw.rect(surface, char_color, pygame.Rect(600, 50, 80, 32), width=30)
two_rect2 = pygame.draw.rect(surface, char_color, pygame.Rect(660, 50, 32, 92), width=30)
two_rect3 = pygame.draw.rect(surface, char_color, pygame.Rect(635, 110, 32, 32), width=30)
two_rect4 = pygame.draw.rect(surface, char_color, pygame.Rect(610, 140, 32, 32), width=30)
two_rect5 = pygame.draw.rect(surface, char_color, pygame.Rect(590, 170, 120, 32), width=30)

# show the map once after drawing obstacles
pygame.display.flip()

robot_radius = 5
clearance = 5
total_clearance = robot_radius + clearance
# ─────────────────────────────────────────────────────────────────────────────
# KEY HELPER: pump events so the window stays responsive while waiting for user input in the terminal
def pump():
    """Call this regularly to keep the pygame window alive."""
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            pygame.quit()
            raise SystemExit

# ─────────────────────────────────────────────────────────────────────────────
# Obstacle check 
def is_free_space(x, y):
    x = int(round(x))
    y = int(round(y))

    # Boundary inflation
    if x < total_clearance or x >= width - total_clearance:
        return False
    if y < total_clearance or y >= height - total_clearance:
        return False

    for dx in range(-total_clearance, total_clearance + 1):
        for dy in range(-total_clearance, total_clearance + 1):
            if dx*dx + dy*dy <= total_clearance*total_clearance:
                nx = x + dx
                ny = y + dy

                color = surface.get_at((nx, ny))[:3]
                if color == obstacle_color or color == char_color:
                    return False

    return True


# ─────────────────────────────────────────────────────────────────────────────
# USER INPUT
def get_valid_coord(label):
    # reads and validates user input state:
    # x y & theta
    while True:
        pump()   # keep window alive while waiting for input
        try:
            x     = int(input(f"[{label}] Enter x: "))
            y     = int(input(f"[{label}] Enter y: "))
            py = height - y # convert Cartesian y → pygame y (inverted)
            theta = int(input(f"[{label}] Enter theta (multiple of 30°): "))
        except ValueError:
            print(" Enter integers only.")
            continue

        if not (0 < x < width):
            print(f" x must be 1–{width-1}")
            continue

        if not (0 < py < height):
            print(f" y must be 1–{height-1}")
            continue

        if theta % 30 != 0:
            print(" Theta must be a multiple of 30.")
            continue

        print("Checking point:", x, py, "theta:", theta)
        
        if not is_free_space(x, py):
            print(" Point is inside an obstacle.")
            continue
        
        # convert user (Cartesian) → pygame
        return (x, py, theta % 360)
    
# -------------------------------------#
# Visited amtrix for duplicates
#---------------------------------------#
#x threshold     = 0.5
#   y threshold     = 0.5
#   theta threshold = 30 degrees
x_bins = int(math.ceil(width / 0.5)) + 1
y_bins = int(math.ceil(height / 0.5)) + 1
V = np.zeros((x_bins, y_bins, 12), dtype=np.uint8)



def state_to_indices(state):
    # converts a state (x, y, theta) to visited-matrix indices

    x, y, theta = state
    i = int(x / .5) # x / .5
    j = int(y / .5) # x / .5 
    k = int((theta % 360) // 30) # theta / .5

    i = max(0, min(i, 499))
    j = max(0, min(j, 1199))
    k = max(0, min(k, 11))
    return i, j, k

def is_visited(state):
    # Returns True if the discretized state region has already been visited.
    i, j, k = state_to_indices(state)
    return V[i, j, k] == 1 # 1 for visited 

def mark_visited(state):
    """
    Marks the discretized state region as visited.
    """
    i, j, k = state_to_indices(state)
    V[i, j, k] = 1


# ─────────────────────────────────────────────────────────────────────────────
# Movement functions
available_turns = [-60, -30, 0, 30, 60]

def apply_action(state, turn_deg, L):
    # Move forward by L after changing heading by turn_deg.
    # Checks intermediate points for collision
    x, y, theta = state
    new_theta = (theta + turn_deg) % 360 # keep orientation 0-359
    rad = math.radians(new_theta) # convert to radians for trig functions
    new_x = x + L * math.cos(rad)
    new_y = y - L * math.sin(rad)#   y-axis flipped in pygame

    # Collision checking each small steps along the path
    steps = max(1, int(L * 2))
    for t in range(steps + 1): # check points along the path at intervals of 0.5 units
        frac = t / steps # fraction of the way along the path
        ix = x + frac * (new_x - x) 
        iy = y + frac * (new_y - y)

        if not is_free_space(int(round(ix)), int(round(iy))): # mid-path obstacle
            return None # invalid move if path hits obstacle

    snapped_theta = int(round(new_theta / 30)) * 30 % 360  # snap theta to nearest 30 deg
   # snap position to 0.5 resolution so it matches visited matrix thresholds
    new_x = round(new_x * 2) / 2
    new_y = round(new_y * 2) / 2
    return (new_x, new_y, snapped_theta)

def get_neighbors(state, L):
    results = []
    for turn in available_turns:
        ns = apply_action(state, turn, L)
        if ns is not None:
            results.append((ns, float(L)))
    return results

def angle_diff(a, b): # Returns the minimum angular difference between a and b.
    diff = abs(a - b) % 360
    return min(diff, 360 - diff)

def is_goal_reached(state, goal_state, pos_threshold=1.5, theta_threshold=30):
    # Checks whether the current state is close enough to the target state
    dx = state[0] - goal_state[0]
    dy = state[1] - goal_state[1]
    dist = math.sqrt(dx*dx + dy*dy)
    dtheta = angle_diff(state[2],  goal_state[2])
    return dist <= pos_threshold and dtheta <= theta_threshold


#---------------#
# Heuristic
#---------------#

def heuristic(state, target_state):
    # for backward A* starting node's the target
    dx = target_state[0] - state[0]
    dy = target_state[1] - state[1]
    dist = math.sqrt(dx * dx + dy * dy)

    dtheta = angle_diff(state[2], target_state[2])

    return dist + 0.5 * (dtheta / 30)


def backward_astar(start_state, goal_state, L):
    search_start = goal_state
    search_target = start_state

    open_heap = []
    parent = {}
    g_score = {}

    g_score[search_start] = 0
    entry_counter = 0

    h0 = heuristic(search_start, search_target)
    heapq.heappush(open_heap, (h0, entry_counter, search_start))

    while open_heap:
        pump()
        _, _, current = heapq.heappop(open_heap)

        # If this discretized state region has already been expanded, skip it
        if is_visited(current):
            continue

        # Mark visited when popped, not when generated
        mark_visited(current)

        # Goal check for backward search:
        # have we reached the original start state?
        if is_goal_reached(current, search_target):
            return parent, current

        # Draw explored node
        x, y, _ = current
        pygame.draw.circle(surface, visited_color, (x, y), 1)

        if entry_counter % 300 == 0:
            pygame.display.update()
            pump()

        current_g = g_score[current]

        for neighbor, move_cost in get_neighbors(current, L):
            if is_visited(neighbor):
                continue

            tentative_g = current_g + move_cost

            if neighbor not in g_score or tentative_g < g_score[neighbor]:
                g_score[neighbor] = tentative_g
                parent[neighbor] = current

                h = heuristic(neighbor, search_target)
                f = tentative_g + h

                entry_counter += 1
                heapq.heappush(open_heap, (f, entry_counter, neighbor))

    return None, None



# ---------------------# 
# Path reconstruction
# --------------------#
def reconstruct_path(parent, reached_node):
    """
    Reconstructs the backward A* path.

    Because the search starts from the goal and moves toward the start,
    the reconstructed path will be:
        near-start -> ... -> goal
    """
    path = [reached_node]
    current = reached_node

    while current in parent:
        current = parent[current]
        path.append(current)

    return path
        

# ─────────────────────────────────────────────────────────────────────────────
# Get user inputs - start, goal, robot radius, clearance, step size
print("Robot radius     :", robot_radius)
print("Clearance        :", clearance)
print("Total inflation  :", total_clearance)

print("Enter START state:")
start_state = get_valid_coord("START")

print("\nEnter GOAL state:")
goal_state  = get_valid_coord("GOAL")


step_size = 0
while not (1 <= step_size <= 10):
    pump()
    try:   
        step_size = int(input("\nStep size L (1–10): "))
    except ValueError: 
        step_size = 0
        pass
    if not (1 <= step_size <= 10):
        print("  step size Must be 1–10.")

L = step_size
print(f"\n✓ Start={start_state}  Goal={goal_state}  L={step_size}\n")


# Run backward A*

start_time = time.time()
parent, reached_state = backward_astar(start_state, goal_state, L)


# the shortest path is discovered using the child to parent dict from before
# the dict is reversed to account for the start of the search beginning at the end of the dict
if reached_state is None:
    print("Search ended without reaching the start state.")
else:
    print("SUCCESS")
    print("Reached state:", reached_state)

    shortest_path = reconstruct_path(parent, reached_state)

    # redraw start and goal first 
    pygame.draw.circle(surface, start_color, (start_state[0], start_state[1]), 5)
    pygame.draw.circle(surface, goal_color, (goal_state[0], goal_state[1]), 5)
 
    # draw final shortest path
    for i in range(len(shortest_path) - 1):
        pump()
        x1, y1, _ = shortest_path[i]
        x2, y2, _ = shortest_path[i + 1]
        pygame.draw.line(surface, path_color, (x1, y1), (x2, y2), 2)
        pygame.display.update()

     # Redraw start and goal so they remain visible
    pygame.draw.circle(surface, start_color, (start_state[0], start_state[1]), 5)
    pygame.draw.circle(surface, goal_color, (goal_state[0], goal_state[1]), 5)
    pygame.display.update()

    print("\nBackward A* Path:")
    for node in shortest_path:
        print(node)

print("--- %s seconds ---" % (time.time() - start_time)) # time for full search and shortest path is recorded for analysis

# keep pygame window open after search so result can be seen
print("Close the pygame window to exit.")

while True:
    pump()
    pygame.display.update()
    pygame.time.delay(30)